In [1]:
import numpy as np
import pandas as pd
import scipy
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import PySimpleGUI as sg
import pyabf
import statistics  # встроенный модуль, не требует установки
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

from scipy.signal import savgol_filter
from scipy.signal import medfilt
from scipy.signal import iirnotch, filtfilt
from scipy.signal import butter, filtfilt
import os
from tkinter import Tk, filedialog

PySimpleGUI is now located on a private PyPI server.  Please add to your pip command: -i https://PySimpleGUI.net/install

The version you just installed should uninstalled:
   python -m pip uninstall PySimpleGUI
   python -m pip cache purge

Then install the latest from the private server:
python -m pip install --upgrade --extra-index-url https://PySimpleGUI.net/install PySimpleGUI

You can also force a reinstall using this command and it'll install the latest regardless of what you have installed currently
python -m pip install --force-reinstall --extra-index-url https://PySimpleGUI.net/install PySimpleGUI

Use python3 command if you're running on the Mac or Linux


In [17]:
def channel (file, search_ch): #gives the time period in every channel to search in
    channels_num = int(file.channelCount)
    
    if channels_num >=2: 
        my_file.setSweep(sweepNumber = 0, channel = 0) #Open 1st sweep to find the stimulus
        stim_line_ch0 = my_file.sweepC #will show only the stimulus (current) where baseline == 0
        stim_step_ch0 = np.array(np.nonzero(stim_line_ch0))[0] # everything what is not 0, is the stim period
        stim_start_ch0 = stim_step_ch0[0] #stim begin
        stim_end_ch0 = stim_step_ch0[-1] #stim stop

    if channels_num == 4: 
        my_file.setSweep(sweepNumber = 0, channel = 1) #channel index is a bog confusion. But sweepC in Ch1 returnes the stim line
        stim_line_ch2 = my_file.sweepC
        stim_step_ch2 = np.array(np.nonzero(stim_line_ch2))[0]
        stim_start_ch2 = stim_step_ch2[0]
        stim_end_ch2 = stim_step_ch2[-1]
        
    points_needed_pre = int(0.125*sample_rate) #calculate the mean value in 125 ms at baseline
    points_needed_in = int(0.25*sample_rate) #calculate the mean value in 250 ms during activity
    points_needed_after = int(0.75*sample_rate) #calculate the mean value in 1.5 s after activity

    if search_ch == 0:
        start = stim_start_ch0
        stop = stim_end_ch0
        pre_start = stim_start_ch0-points_needed_pre
        post_stop = stim_end_ch0+points_needed_after
        within_point = stim_end_ch0-points_needed_in

    if search_ch == 2:
        start = stim_start_ch2
        stop =  stim_end_ch2
        pre_start = stim_start_ch2-points_needed_pre
        post_stop = stim_end_ch2+points_needed_after
        within_point = stim_end_ch2-points_needed_in
        
    return start, stop, pre_start, post_stop, within_point


def infl_points (file, sweep, search_ch, start, stop):
        file.setSweep(sweep, channel = search_ch)
        smooth = gaussian_filter1d(file.sweepY[start:stop], 30) # smooth, large window for low-frequency signals
        d1 = np.gradient(smooth)
        #d2 = np.gradient(d1) # compute second derivative
        smooth_d1 = gaussian_filter1d(d1, 200)
        
        d2 = np.gradient(smooth_d1)
        smooth_d2 = gaussian_filter1d(d2, 5)
        
        infls = np.where(np.diff(np.sign(smooth_d2)))[0] # find switching points
        ind_infls = infls + start
        
        return ind_infls, smooth, d1, d2, smooth_d1, smooth_d2
    
    
def infl_points_startAP (file, sweep, search_ch, start, stop):
    #threshold = 0.20
    file.setSweep(sweep, channel = search_ch)
    smooth = gaussian_filter1d(file.sweepY[start:stop], 2) # smooth
    d1 = np.gradient(smooth)
    d2 = np.gradient(d1) # compute second derivative
    #
    #smooth_d1 = gaussian_filter1d(d1, 205)
    #smooth_d2 = gaussian_filter1d(d2, 205)
    infls = np.where(np.diff(np.sign(d2)))[0]# find switching points
    #cross_up = np.where((d1[:-1] < threshold) & (d1[1:] >= threshold))[0]
    ind_infls = infls + start
    #ind_infls = cross_up + start

    return ind_infls

def spike_begin_threshold(ind_peaks, ind_infls):
    start_ap_ind = []
    end_ap_ind = []
    
    if len(ind_peaks) == 0 or len(ind_infls) == 0:
        return start_ap_ind, end_ap_ind

    for peak in ind_peaks: 

        infl_before_peak = [i for i in ind_infls if i < peak] # Найти индексы точек перегиба, которые меньше пика

        if len(infl_before_peak) >= 1: # Берём ПРЕДпоследнюю перед пиком
            #start_ap_ind.append(infl_before_peak[-2])
            start_ap_ind.append(infl_before_peak[-1])
        else: # Если перегиб только один или ни одного — добавляем np.nan
            start_ap_ind.append(np.nan)

        infl_after_peak = [i for i in ind_infls if i > peak]

        if len(infl_after_peak) >= 1: # Берём вторую после пика пиком, в идеале дб в минимуме?
            end_ap_ind.append(infl_after_peak[0])
        else: # Если перегиб только один или ни одного — добавляем np.nan
            end_ap_ind.append(np.nan)

    return start_ap_ind, end_ap_ind

def spike_begin(ind_peaks, ind_infls):
    #Not real mathematically correctly. It will be a point, where the d2 is not "noisy" anymore
    start_ap_ind = []
    end_ap_ind = []
    
    if len(ind_peaks) == 0 or len(ind_infls) == 0:
        return start_ap_ind, end_ap_ind

    for peak in ind_peaks: 

        infl_before_peak = [i for i in ind_infls if i < peak] # Найти индексы точек перегиба, которые меньше пика

        if len(infl_before_peak) >= 2: # Берём ПРЕДпоследнюю перед пиком
            start_ap_ind.append(infl_before_peak[-2])
        else: # Если перегиб только один или ни одного — добавляем np.nan
            start_ap_ind.append(np.nan)

        infl_after_peak = [i for i in ind_infls if i > peak]

        if len(infl_after_peak) >= 2: # Берём вторую после пика пиком, в идеале дб в минимуме?
            end_ap_ind.append(infl_after_peak[1])
        else: # Если перегиб только один или ни одного — добавляем np.nan
            end_ap_ind.append(np.nan)

    return start_ap_ind, end_ap_ind
    
    
def peak (file, sweep, search_ch, start, stop, v_level= -10): #find indexes of peaks 
    #v_level gives the threashold for spike detection
    if not np.isnan(sweep):
        file.setSweep(sweep, channel = search_ch)
        peaks= find_peaks(file.sweepY[start:stop], height=v_level, distance=10)
        ind_peaks = peaks[0] + start
        return ind_peaks #else return None
    return [] 
    
def begin_RB (smoth_file, start, stop):
    min_index = np.argmin(smoth_file[0:stop-start+1])+start #min of d1 with correct indexing
    return min_index

def ind_lin_fit (smooth_d1, smooth_d2, start, stop, onset):
    #set thresholds for both derivatives, for almost horizontal lines both should be close to 0
    eps_1 = 0.0015 # for d1
    eps_2 = 0.000005   # for d2
    
    #find index where both d1 and d2 are close to 0
    linear_candidate_local = np.where((np.abs(smooth_d2[start-onset:stop-onset]) < eps_2)
                            & (np.abs(smooth_d1[start-onset:stop-onset]) < eps_1))[0][0]
    linear_candidate = linear_candidate_local + start
    return linear_candidate
                            
def frequencies (ind_peaks, sampling_rate):
    freq_list = [sampling_rate/(ind_peaks[i+1]-ind_peaks[i]) for i in range (0, len(ind_peaks)-1)]
    
    if len(ind_peaks)>=2:
        initial_freq = np.round(freq_list[0], 3) #in Hz
    else:
        initial_freq = np.nan
        
    if len(ind_peaks)>=3:
        mean_freq = np.round(statistics.mean(freq_list[1:]), 3)#Mean firing frequency Hz
    else:
        mean_freq = np.nan
        
    freq = np.round(statistics.mean(freq_list[:]), 3)
    
    return initial_freq, mean_freq, freq

def min_after_spike (file, sweep, search_ch, start, sampling_rate):
    file.setSweep(sweep, channel = search_ch)
    my_period = 0.010 #10 ms
    end = start + int(sampling_rate*my_period)
    segment = file.sweepY[start:end]
    local_min_index = np.argmin(segment)
    global_min_index = start + local_min_index
    return global_min_index


def peak_amplitudes_ratio (file, sweep, search_ch, ap_start, ap_peak): #AP ratio calculation (Currently only 1st and 2nd)
    if not np.isnan(sweep):
        file.setSweep(sweep, channel = search_ch)
        amplitudes = [file.sweepY[peak] - file.sweepY[start] for peak, start in zip(ap_peak, ap_start)]

        # Отношение 2/1 (если доступно)
        ampl_ratio_21 = np.round(amplitudes[1] / amplitudes[0], 2) if len(amplitudes) >= 2 else np.nan

        # Список отношений i+1 / i начиная с 2 (т.е. 3/2, 4/3 и т.д.)
        successive_ratios = [np.round(amplitudes[i+1] / amplitudes[i], 2) 
                             for i in range(1, len(amplitudes)-1)]

        # Среднее значение этих отношений
        mean_successive_ratio = np.round(np.mean(successive_ratios), 2) if successive_ratios else np.nan

        return ampl_ratio_21, successive_ratios, mean_successive_ratio
    return np.nan, [], np.nan

def IV_values(file, sweep, voltage_channel, current_channel,
              pre_start, pre_stop, 
              in_start, in_stop):
    if not np.isnan(sweep):
        file.setSweep(sweep, channel = voltage_channel)
        voltage_pre_stim = np.round(np.mean(file.sweepY[pre_start:pre_stop]), 3) #the cell was held at...
        voltage_in_stim = np.round(np.mean(file.sweepY[in_start:in_stop]), 3)
        file.setSweep(sweep, channel = current_channel)
        current_pre_stim = np.mean(file.sweepY[pre_start:pre_stop]) 
        current_in_stim = np.mean(file.sweepY[in_start:in_stop])
        current_injection = np.round(current_in_stim - current_pre_stim, 3) # the applied current where more than 4 APs
        return voltage_pre_stim, voltage_in_stim, current_injection
    return np.nan, np.nan, np.nan

In [11]:
folder_path = r'C:\Users\user\Documents\IMBIT\TRN RBs'

In [10]:
# Задаем пустую таблицу
# Определи названия столбцов
columns = ['File Name', 
           'width_LTS, ms', 
           'number_spikes per LTS', 
           'Initial Firing Frequency [0:1], Hz',
           'Firing Frequency ALL [:], Hz', 
           'Firing Frequency [1:], Hz', 
           'Voltage Level, mV', 
           'Amplitude 1, mV', 
           'LTS Amplitude, mV', 
           'Amplitude Ratio 12', 
           'Hold_voltage, mV', 
           'Current inj, pA', 
           'Membrane Potential, mV'
          ]

# Создай пустой DataFrame с этими столбцами (без данных)
df = pd.DataFrame(columns=columns)

# Сохрани в Excel
excel_path = f'{folder_path}\\results.xlsx'
df.to_excel(excel_path, index=False)

print("Пустая таблица создана: results.xlsx")

Пустая таблица создана: results.xlsx


In [12]:
my_sweep = 0
#my_search_ch = int(input("Please enter your #channel (0 or 2!) "))
my_search_ch = 0


In [13]:
# Выбираем папку!
root = Tk()
root.withdraw()
root.attributes('-topmost', True)

# Ask the user to select a folder
folder_path_all = filedialog.askdirectory(
    title="Выберите папку"
)

print("Выбрана папка:", folder_path_all)

Выбрана папка: F:/TRN/TRN RBs


In [14]:
for file_name in os.listdir(folder_path_all):
    if file_name.endswith(".abf"):
        file_path = os.path.join(folder_path_all, file_name)
        
        try:

            # Открываем ABF-файл
            my_file = pyabf.ABF(file_path)
            filename = my_file.abfID  # без расширения .abf
            

            print(f"Обрабатывается файл: {filename}")
            
            channels_num = int(my_file.channelCount)
            sample_rate = int(my_file.dataRate)

            my_start, my_stop, my_pre_start, my_post_stop, my_within_point = channel(my_file, my_search_ch) #returns stimulation start and stop in the chosen
            my_ind_infls, smoothed_file, my_d1, my_d2, my_smooth_d1, my_smooth_d2 = infl_points (my_file, my_sweep, my_search_ch, my_stop, my_post_stop)
            my_ind_peaks = peak (my_file, my_sweep, my_search_ch, my_stop, my_post_stop) # find peaks' indexes
            my_min_index = begin_RB (my_smooth_d1, my_stop, my_ind_peaks[0]) #min of d1 with correct indexing (inflection point/start of RB event)
            my_ind_infls_AP = infl_points_startAP (my_file, my_sweep, my_search_ch, my_stop, my_post_stop) #find all inflection points
            my_start_ap_ind, my_end_ap_ind = spike_begin(my_ind_peaks, my_ind_infls_AP) # Find start and end of spikes
            my_linear_candidate = ind_lin_fit (my_smooth_d1, my_smooth_d2, my_ind_peaks[-1], my_post_stop, my_stop)


            voltage_level = np.mean([my_file.sweepY[my_start_ap_ind]])

            width_LTS = np.round(((my_linear_candidate - my_min_index)/sample_rate)*1000, 3) #in ms
            number_spikes = len(my_ind_peaks)
            burst_freq_12, burst_freq, burst_freq_all = frequencies (my_ind_peaks, sample_rate) #in Hz
            ampl_1 = np.round(np.abs(my_file.sweepY[my_ind_peaks[0]]-my_file.sweepY[my_start_ap_ind[0]]), 2) #in mV

            last_spike_end = min_after_spike (my_file, my_sweep, my_search_ch, my_ind_peaks[-1], sample_rate)
            LTS_hight = np.mean([my_file.sweepY[my_start_ap_ind[0]], my_file.sweepY[last_spike_end]])
            LTS_amp = np.round(np.abs(LTS_hight-my_file.sweepY[my_min_index]), 3)
            ap_ampl_ratio, my_successive_ratios, my_mean_successive_ratio = peak_amplitudes_ratio (my_file, my_sweep, my_search_ch, 
                                                                                                   my_start_ap_ind, my_ind_peaks)

            hold_voltage, membrane_voltage, inj_current = IV_values(my_file, my_sweep, my_search_ch, my_search_ch + 1, 
                                             my_pre_start, my_start, my_within_point, my_stop)

            my_file.setSweep(my_sweep, channel = my_search_ch)

            fig = plt.figure(figsize=(40, 80))
            plt.subplot(3, 1, 1)
            plt.suptitle(f"RB Signal: {filename}", fontsize=20)

            plt.plot(my_file.sweepX[my_stop:my_post_stop], my_file.sweepY[my_stop:my_post_stop], label='Original/Noisy Data', linewidth=1)
            #plt.plot(my_file.sweepX[my_stop:my_post_stop], smoothed_file, label='Filtered Data', linewidth=1)
            #plt.scatter(my_file.sweepX[my_ind_infls], my_file.sweepY[my_ind_infls])
            plt.scatter(my_file.sweepX[my_ind_peaks], my_file.sweepY[my_ind_peaks], color='red', label='Peaks', s=100)
            plt.scatter(my_file.sweepX[my_min_index], my_file.sweepY[my_min_index], color='green', label='RB Start', s=100)
            #plt.scatter(my_file.sweepX[my_ind_infls_AP], my_file.sweepY[my_ind_infls_AP])
            plt.scatter(my_file.sweepX[my_start_ap_ind], my_file.sweepY[my_start_ap_ind], color='blue', label='AP Start', s=100)
            #plt.scatter(my_file.sweepX[my_end_ap_ind], my_file.sweepY[my_end_ap_ind])
            plt.scatter(my_file.sweepX[last_spike_end], my_file.sweepY[last_spike_end], color='black', label='AP last End', s=100)
            plt.scatter(my_file.sweepX[my_linear_candidate], my_file.sweepY[my_linear_candidate], color='magenta', label='RB End', s=100)
            #plt.scatter(my_file.sweepX[crossing_index], my_file.sweepY[crossing_index])
            plt.hlines(LTS_hight, my_file.sweepX[my_stop], my_file.sweepX[my_post_stop], linestyle='dotted', linewidth=0.5, label='LTS Hight')
            plt.hlines(voltage_level, my_file.sweepX[my_stop], my_file.sweepX[my_post_stop], linestyle='dotted', linewidth=0.5, label='Spike Start Level')
            #plt.vlines(my_file.sweepX[my_ind_peaks[-1]], -50, 20)
            #plt.vlines(my_file.sweepX[end], -50, 20)
            #plt.hlines(voltage_level+threshold_y, my_file.sweepX[my_stop], my_file.sweepX[my_post_stop])
            plt.xlim(my_file.sweepX[my_stop], my_file.sweepX[my_post_stop])
            plt.legend(fontsize=35)

            #plt.ylim(-60, -35)

            plt.subplot(3, 1, 2)
            plt.suptitle(f"D1: {filename}", fontsize=20)
            plt.plot(my_file.sweepX[my_stop:my_post_stop], my_smooth_d1, label='Filterd D2', linewidth=1)
            plt.plot(my_file.sweepX[my_stop:my_post_stop], my_d1, label='Noisy D1', linewidth=1)
            plt.scatter(my_file.sweepX[my_min_index], my_smooth_d1[my_min_index-my_stop], color='green', label='Min', s=100)
            plt.vlines(my_file.sweepX[my_linear_candidate], -0.02, 0.02, linestyle='dotted', linewidth=0.5, label='Linear Fit Start')
            #plt.ylim(-1, 1)
            plt.ylim(-0.02, 0.02)
            plt.xlim(my_file.sweepX[my_stop], my_file.sweepX[my_post_stop])
            plt.legend(fontsize=35)

            plt.subplot(3, 1, 3)
            plt.suptitle(f"D2: {filename}", fontsize=20)
            plt.plot(my_file.sweepX[my_stop:my_post_stop], my_smooth_d2, label='Filtered D2', linewidth=1)
            plt.plot(my_file.sweepX[my_stop:my_post_stop], my_d2, label='Noisy D2', linewidth=1)
            plt.hlines(0.000005, my_file.sweepX[my_stop], my_file.sweepX[my_post_stop], color='green', linestyle='dotted', linewidth=1, label='Linear Fit Area')
            plt.hlines(-0.000005, my_file.sweepX[my_stop], my_file.sweepX[my_post_stop], color='green', linestyle='dotted', linewidth=1)
            plt.hlines(0, my_file.sweepX[my_stop], my_file.sweepX[my_post_stop], color='black', linestyle='dotted', linewidth=1)
            #plt.ylim(-0.02, 0.02)
            plt.ylim(-0.00005, 0.00005)
            plt.xlim(my_file.sweepX[my_stop], my_file.sweepX[my_post_stop])
            plt.legend(fontsize=35)


            #plt.tight_layout()


            save_path = os.path.join(folder_path, filename)
            plt.savefig(save_path)
            plt.close()

            new_values = [filename, 
                  width_LTS, 
                  number_spikes, 
                  burst_freq_12, 
                  burst_freq_all, 
                  burst_freq, 
                  voltage_level, 
                  ampl_1, 
                  LTS_amp, 
                  ap_ampl_ratio, 
                  hold_voltage, 
                  inj_current, 
                  membrane_voltage
                 ]


            df = pd.read_excel(excel_path)
            new_row = pd.Series(new_values, index=df.columns)
            df = pd.concat([df, new_row.to_frame().T], ignore_index=True)
            df.to_excel(excel_path, index=False)
            
        except Exception as e:
                print(f"Ошибка при обработке файла {file_name}: {e}")
                continue  # перейти к следующему файлу

Обрабатывается файл: 2024_04_23_0001
Обрабатывается файл: 2024_11_21_0000
Обрабатывается файл: 2024_11_21_0007
Ошибка при обработке файла 2024_11_21_0007.abf: mean requires at least one data point
Обрабатывается файл: 2024_11_21_0013
Обрабатывается файл: 2025_04_03_0002
Ошибка при обработке файла 2025_04_03_0002.abf: index 0 is out of bounds for axis 0 with size 0
Обрабатывается файл: 2024_04_25_0013
Ошибка при обработке файла 2024_04_25_0013.abf: mean requires at least one data point
Обрабатывается файл: 2024_04_29_0027
Ошибка при обработке файла 2024_04_29_0027.abf: index 0 is out of bounds for axis 0 with size 0
Обрабатывается файл: 2024_04_29_0036
Обрабатывается файл: 2024_05_01_0002
Ошибка при обработке файла 2024_05_01_0002.abf: mean requires at least one data point
Обрабатывается файл: 2024_05_01_0005
Обрабатывается файл: 2024_05_01_0008
Обрабатывается файл: 2024_05_01_0018
Обрабатывается файл: 2024_05_01_0026
Обрабатывается файл: 2024_05_06_0046
Ошибка при обработке файла 2024_